# CIDA-Swarm — рой, который думает вместе

Два слоя коллективного интеллекта:
* **общая карта** (стигмергия) — где рой уже был, как феромон у муравьёв;
* **CIDA-связь** — дроны в радиусе связи обмениваются сообщениями о том, где видели людей и что оказалось
  приманкой; громче звучит тот, чья карта «где люди» сильнее расходится с остальными.

Сравниваем рой **с общей картой** (`stigmergy_s0`, ноутбук `CIDA_Swarm_Step1`) и рой **с общей картой + CIDA**
на одной и той же сцене. Runtime → **A100**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys
if not os.path.exists("/content/CIDA-plugin"):
    !git clone https://github.com/Kairatzh/CIDA-plugin.git /content/CIDA-plugin
%cd /content/CIDA-plugin
!git pull --ff-only
!git log --oneline -1
from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/cida_swarm_runs_v2"
os.makedirs(RUNS, exist_ok=True)
print("рой с общей картой найден:", os.path.exists(f"{RUNS}/stigmergy_s0/model.pt"))

## 1. Обучение CIDA-роя (~20–30 минут)

Тот же мир и та же общая карта, что у `stigmergy_s0`; разница только в CIDA-связи. Бюджет 1,5 млрд шагов:
базовые рои выходили на плато примерно к 1,3 млрд.

In [ ]:
RUN = "cida_stig_s0"
cmd = (f"{sys.executable} -u -m cida_swarm.train --comm cida --share-coverage --n-envs 16384 --total-steps 1.5e9 "
       f"--eval-every 25 --seed 0 --name {RUN} --out {RUNS}/{RUN}")
if os.path.exists(f"{RUNS}/{RUN}/result.json"):
    print("уже обучен — пропускаю")
else:
    print(">>>", cmd)
    get_ipython().system(cmd)

## 2. Все рои в цифрах

In [ ]:
import json
rows = []
for name in ["nocomm_s0", "stigmergy_s0", "cida_stig_s0"]:
    path = f"{RUNS}/{name}/result.json"
    if os.path.exists(path):
        r = json.load(open(path))
        f = r["final"]
        rows.append((name, f["found_frac"], f["all_found"], f["length"], f["coverage"], r["train_minutes"]))
print("| Рой | Нашли людей | Нашли всех | Шагов на миссию | Исследовано | Минут обучения |")
print("|---|---|---|---|---|---|")
for name, found, all_found, length, cov, minutes in rows:
    print(f"| {name} | {found:.1%} | {all_found:.1%} | {length:.0f} | {cov:.1%} | {minutes:.0f} |")

## 3. Анимация: одна сцена, два роя

Слева рой с общей картой, справа тот же рой + CIDA. Фиолетовые линии — кто кого слушает (толще = важнее
сообщение), красная заливка — общее убеждение роя «где люди». Зелёные точки — люди, оранжевые ромбы — приманки.

In [ ]:
from IPython.display import Image, display
for seed in [0, 1, 2]:
    out = f"{RUNS}/compare_scene{seed}.gif"
    !{sys.executable} -m cida_swarm.render {RUNS}/stigmergy_s0/model.pt {RUNS}/cida_stig_s0/model.pt         --labels "общая карта" "общая карта + CIDA" --seed {seed} --n-drones 6 --out {out}
    display(Image(filename=out))

## Что дальше

* Если рой с CIDA находит больше людей или быстрее — это и есть демо: те же дроны, но они **договариваются**.
  Если нет — значит, CIDA-связь в этом виде не нужна, и это стоит знать до демо, а не после.
* Анимации лежат на Google Drive (`compare_scene*.gif`).
* Следующий шаг продукта — «нужна помощь», совместный подъём и браузерное демо, где зритель сам прячет людей
  и сбивает дронов.